# Experiment 2: Activation functions and their centralizers

The centralizer of an activation $\sigma$ is the set of invertible matrices $M$ satisfying $\sigma(Mz) = M\sigma(z)$ for all $z$. Different activations yield different centralizers:

- **ReLU**: $\mathrm{Cent}(\mathrm{ReLU}) \cong \mathcal{D}^+ \rtimes S_n$ (positive diagonal scaling + permutations)
- **Sigmoid**: $\mathrm{Cent}(\mathrm{sigmoid}) = S_n$ (permutations only)

We verify these claims empirically by testing which matrix types commute with each activation.


In [ ]:
import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'figure.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
})

torch.manual_seed(42)
np.random.seed(42)

n = 5  # dimension
n_samples = 1000  # random z vectors to average over

def commutation_error(sigma, M, n_samples=1000):
    """Compute mean ||sigma(Mz) - M*sigma(z)|| over random z vectors."""
    z = torch.randn(n_samples, n, dtype=torch.float64)
    lhs = sigma(z @ M.T)        # sigma(Mz) for each row
    rhs = sigma(z) @ M.T        # M*sigma(z) for each row
    return torch.norm(lhs - rhs, dim=1).mean().item()

relu = torch.relu
sigmoid = torch.sigmoid

print("Setup complete. Testing commutation errors...")


## 1. Testing matrix types against ReLU and sigmoid

We test four types of matrices:
1. Positive diagonal $D^+$ (should commute with ReLU, not sigmoid)
2. Permutation $P$ (should commute with both)
3. Negative scaling $-\alpha I$ (should commute with neither)
4. General invertible $M$ (should commute with neither)


In [ ]:
def random_positive_diagonal(n):
    d = torch.abs(torch.randn(n, dtype=torch.float64)) + 0.1
    return torch.diag(d)

def random_permutation(n):
    perm = torch.randperm(n)
    P = torch.zeros(n, n, dtype=torch.float64)
    P[torch.arange(n), perm] = 1.0
    return P

def random_negative_scaling(n):
    alpha = torch.abs(torch.randn(1, dtype=torch.float64)).item() + 0.5
    return -alpha * torch.eye(n, dtype=torch.float64)

def random_invertible(n):
    M = torch.randn(n, n, dtype=torch.float64)
    while torch.linalg.det(M).abs() < 0.1:
        M = torch.randn(n, n, dtype=torch.float64)
    return M

# Run trials
n_trials = 50
matrix_types = ['Positive diagonal', 'Permutation', 'Negative scaling', 'General invertible']
generators = [random_positive_diagonal, random_permutation, random_negative_scaling, random_invertible]

results = {act_name: {mt: [] for mt in matrix_types} 
           for act_name in ['ReLU', 'Sigmoid']}

for act_name, sigma in [('ReLU', relu), ('Sigmoid', sigmoid)]:
    for mt, gen in zip(matrix_types, generators):
        for _ in range(n_trials):
            M = gen(n)
            err = commutation_error(sigma, M, n_samples)
            results[act_name][mt].append(err)

# Print summary table
print(f"{'Matrix type':<22} | {'ReLU error':>14} | {'Sigmoid error':>14}")
print("-" * 56)
for mt in matrix_types:
    relu_mean = np.mean(results['ReLU'][mt])
    sig_mean = np.mean(results['Sigmoid'][mt])
    print(f"{mt:<22} | {relu_mean:>14.6f} | {sig_mean:>14.6f}")


## 2. Detailed verification

Let us verify each claim from the article with explicit examples.


In [ ]:
# Detailed verification with a single z vector
z = torch.randn(n, dtype=torch.float64)

print("=== ReLU commutation tests ===\n")

# Positive diagonal: should commute
D = random_positive_diagonal(n)
print(f"Positive diagonal D (entries: {torch.diag(D).numpy().round(3)})")
print(f"  ReLU(Dz) = {relu(D @ z).numpy().round(6)}")
print(f"  D*ReLU(z) = {(D @ relu(z)).numpy().round(6)}")
print(f"  Difference: {torch.norm(relu(D @ z) - D @ relu(z)).item():.2e}\n")

# Permutation: should commute
P = random_permutation(n)
print(f"Permutation P")
print(f"  ReLU(Pz) = {relu(P @ z).numpy().round(6)}")
print(f"  P*ReLU(z) = {(P @ relu(z)).numpy().round(6)}")
print(f"  Difference: {torch.norm(relu(P @ z) - P @ relu(z)).item():.2e}\n")

# Negative scaling: should NOT commute
alpha = 2.0
print(f"Negative scaling: -alpha*I with alpha={alpha}")
neg_M = -alpha * torch.eye(n, dtype=torch.float64)
print(f"  ReLU(-alpha*z) = {relu(neg_M @ z).numpy().round(6)}")
print(f"  -alpha*ReLU(z) = {(neg_M @ relu(z)).numpy().round(6)}")
print(f"  Difference: {torch.norm(relu(neg_M @ z) - neg_M @ relu(z)).item():.4f}")
print(f"  (ReLU kills negatives, so ReLU(-alpha*z_i) = 0 but -alpha*ReLU(z_i) = -alpha*max(0,z_i))\n")

print("\n=== Sigmoid commutation tests ===\n")

# Positive diagonal: should NOT commute with sigmoid
D = random_positive_diagonal(n)
alpha_val = torch.diag(D)[0].item()
print(f"Positive diagonal D (first entry: {alpha_val:.3f})")
print(f"  sigmoid(Dz)[0] = {sigmoid(D @ z)[0].item():.6f}")
print(f"  D*sigmoid(z)[0] = {(D @ sigmoid(z))[0].item():.6f}")
print(f"  Difference: {torch.norm(sigmoid(D @ z) - D @ sigmoid(z)).item():.4f}")
print(f"  (sigmoid compresses nonlinearly, so sigmoid(alpha*t) != alpha*sigmoid(t))\n")

# Permutation: should commute with sigmoid
P = random_permutation(n)
print(f"Permutation P")
print(f"  sigmoid(Pz) = {sigmoid(P @ z).numpy().round(6)}")
print(f"  P*sigmoid(z) = {(P @ sigmoid(z)).numpy().round(6)}")
print(f"  Difference: {torch.norm(sigmoid(P @ z) - P @ sigmoid(z)).item():.2e}")


## 3. Summary

Mean commutation error $\|\sigma(Mz) - M\sigma(z)\|$ for each combination of activation and matrix type. Near-zero means the matrix type is in the centralizer.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

relu_means = [np.mean(results['ReLU'][mt]) for mt in matrix_types]
sig_means = [np.mean(results['Sigmoid'][mt]) for mt in matrix_types]

x_pos = np.arange(len(matrix_types))
width = 0.35

# Clamp zeros to a small value for log scale
relu_plot = [max(v, 1e-16) for v in relu_means]
sig_plot = [max(v, 1e-16) for v in sig_means]

ax.bar(x_pos - width/2, relu_plot, width, label='ReLU',
       color='#B8D4E3', edgecolor='white', linewidth=0.5)
ax.bar(x_pos + width/2, sig_plot, width, label='Sigmoid',
       color='#C47A5A', edgecolor='white', linewidth=0.5)

ax.set_yscale('log')
ax.set_xlabel('Matrix type')
ax.set_ylabel('Mean commutation error')
ax.set_title('Commutation error: which matrices commute with which activations?')
ax.set_xticks(x_pos)
ax.set_xticklabels(matrix_types, rotation=15, ha='right')
ax.legend()

ax.axhline(y=1e-14, color='#B8D4E3', linestyle='--', alpha=0.5)
ax.text(len(matrix_types) - 1, 2e-14, 'machine precision', ha='right',
        fontsize=9, color='#B8D4E3', style='italic')

plt.tight_layout()
plt.savefig('Ex2_fig_activation_centralizer.png', dpi=200, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print("Figure saved: Ex2_fig_activation_centralizer.png")


## Summary

| Matrix type | In Cent(ReLU)? | In Cent(Sigmoid)? |
|---|---|---|
| Positive diagonal $\mathcal{D}^+$ | Yes | No |
| Permutation $S_n$ | Yes | Yes |
| Negative scaling | No | No |
| General invertible | No | No |

This confirms the subgroup chain: $S_n \subsetneq \mathcal{D}^+ \rtimes S_n \subsetneq \mathrm{GL}(n)$. Sigmoid's centralizer is strictly smaller than ReLU's, meaning sigmoid encodes a stronger structural prior.
